# M4 진입 대기 복원 — FIXED2 (실패해도 전부 검사하고 원인을 자세히 기록)

저장 상태를 다시 불러왔을 때 진입 대기 차량 한 대가 목록에 나타나지 않아 검사가 멈췄던 문제(FIXED와 같은 수정)를 포함하고, **실패했을 때 무엇이 달랐는지 자세히 남기고 나머지 상황도 끝까지 검사**합니다.

1. 이 파일을 CPU 런타임으로 열고 **런타임 → 모두 실행**을 누릅니다. (오래된 RESTORE 노트북이나 FIXED 노트북이 실행 중이면 먼저 중지하세요.)
2. 먼저 예전에 실패한 상황(20220812 07:30 난수 43)을 검사합니다. 몇 분 안에 `통과` 또는 `다시 실패`가 보입니다.
3. 통과해도 실패해도 나머지 15개 상황을 모두 검사한 뒤 **통과·실패 요약(restore_report.json)**을 보여 줍니다.
4. 전부 통과하면 자동으로 후보 비교(롤아웃)로 이어갑니다. 하나라도 실패하면 거기서 멈추고 아무것도 버리지 않습니다.
5. 실패하면 이 노트북의 **마지막 셀 출력 전체**를 그대로 알려 주세요. 필요하면 Drive `explore_m4probe_pending_v1/restore_diagnostics/`의 파일을 확인합니다.

기존 결과는 보존합니다. 작업시간·계산량 제한은 두지 않습니다. 통과 기준(관측 110칸·모드 행동·마스크·실행 현시·차량/신호/대기 지문이 300초 동안 매 5초 같을 것)은 FIXED와 똑같고 조금도 완화하지 않았습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE_ROOT / "FutureTraffic_M4PROBE_FIXED2.zip"
SOURCE = DRIVE_ROOT / "explore_weight_v1"
FIXED = DRIVE_ROOT / "explore_fixed24_v1"
NIGHT = DRIVE_ROOT / "explore_night_v1"
OUT = DRIVE_ROOT / "explore_m4probe_pending_v1"
WORKSPACE = Path("/content/FutureTraffic_M4PROBE_FIXED2")
WORK = Path("/content/m4probe_pending_work")
for path in (ARCHIVE, SOURCE / "experiment.json", FIXED / "experiment.json", FIXED / "budget.json", NIGHT / "experiment.json"):
    assert path.is_file(), f"필요한 파일이 없습니다: {path}"
print("CPU 개수:", os.cpu_count() or 1, "| 새 결과 폴더:", OUT)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib
import json
import stat
import sys
import types
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "M4_PROBE_PENDING_FIX_V2", "M4 FIXED2 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델은 Drive 원본에서 읽어야 합니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("m4probe_fixed2_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE}")

In [ ]:
import subprocess
import importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}
source_packages = json.loads((SOURCE / "experiment.json").read_text())["packages"]
assert all(source_packages.get(name) == version for name, version in EXPECTED.items()), "출발 실험의 패키지 버전이 이 묶음과 다릅니다."

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("출발 실험과 패키지 버전이 모두 같아 재설치를 건너뜁니다.")
else:
    print("출발 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 출발 실험과 다릅니다: {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
import subprocess
import signal
from IPython.display import display, Markdown
ENV = {**os.environ, "FOCUSED_RULESET": "pet1", "FOCUSED_NORTH_EXT": "240", "PYTHONUNBUFFERED": "1",
       "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
def run_live(args):
    process = subprocess.Popen([sys.executable, "-u", *args], cwd=WORKSPACE, env=ENV,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    try:
        for line in process.stdout:
            print(line.rstrip())
        code = process.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=20)
        except ProcessLookupError:
            pass
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
        raise
    if code != 0:
        raise RuntimeError(f"작업이 중단됐습니다 (종료 코드 {code}). 위의 원인을 확인해 주세요.")

for pattern in ("test_explore_m4probe_fixed2.py", "test_explore_m4probe_pending.py", "test_explore_m4probe_repaired.py"):
    run_live(["-m", "unittest", "discover", "-s", "tests", "-p", pattern])
print("실행 전 검사 통과")

In [ ]:
# 이전에 실패한 상태를 먼저 검증하고, 통과 여부와 상관없이 나머지 상황도 모두 검사합니다.
# 전부 통과하면 후보 비교로 이어가고, 하나라도 실패하면 원인을 자세히 남기고 멈춥니다.
import json
try:
    run_live(["src/explore_m4probe_fixed2.py", "--out", str(OUT), "--work", str(WORK),
              "--source", str(SOURCE), "--fixed", str(FIXED), "--night", str(NIGHT), "--workers", "0"])
finally:
    report = OUT / "restore_report.json"
    if report.is_file():
        data = json.loads(report.read_text())
        print("복원 검증 요약:", data["summary"])
        for row in data["scenarios"]:
            if row["status"] != "ok":
                print(" -", row["scenario"], row["status"], row.get("error"))
    summary = OUT / "summary.md"
    if summary.is_file():
        display(Markdown(summary.read_text()))
    print("복원 진단 폴더:", OUT / "restore_diagnostics")
    print("실패한 상태 보존 폴더:", OUT / "restore_failed")
    print("결과 폴더:", OUT)
